# 02 - Nettoyage des données DPE

**Objectif** : préparer des données propres pour les départements 59, 69 et 75.
Chaque département est traité séparément, avec les mêmes règles appliquées à tous.

## 1. Chargement
Les données brutes du snapshot `2026-10-12_132605` (neufs et existants, par département) sont chargées, puis fusionnées par département dans un dictionnaire : une entrée par département.

> **Note de mise à jour (octobre 2026)** — ce notebook a été écrit en deux temps.
>
> - L'**exploration** (étapes 1 à 14) a d'abord été faite sur un snapshot plus petit : 10 000 lignes par type et par département, soit 58 644 lignes. Les commentaires « Observations » citent encore ses chiffres (Lille, Lyon, Paris). Le code a ensuite été relancé sur le snapshot `2026-10-12_132605` (100 000 logements existants par département) : **les sorties affichées sont celles de ce snapshot et peuvent différer des commentaires**. Les décisions prises restent valables, avec les évolutions ci-dessous.
> - **Surface minimale : 5 m²** (et non 9 m²). À 9 m², le nettoyage retirait à Paris environ 1 170 vrais petits logements (chambres de service, studios).
> - **Le nettoyage final ne sélectionne plus de colonnes** : toutes sont conservées, sauf celles dont plus de 80 % des valeurs sont manquantes. La sélection de 30 colonnes (section 13) est l'historique de l'exploration.
> - **Les valeurs manquantes ne sont plus remplacées** (ni « non applicable », ni énergies regroupées, ni suppression des lignes incomplètes) : le remplissage se fera à la modélisation, sur l'entraînement seulement.
> - Les chiffres du snapshot actuel sont dans la section 15 et la conclusion (section 16). Le détail est dans `docs/nettoyage_donnees.md`.


In [2]:
import sys
print(sys.executable)

c:\Users\albar\Documents\GitHub\SISE-CONSO-ANALYSIS\.venv\Scripts\python.exe


In [3]:
import sys
sys.path.append("..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocessing import DataPreprocessor

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 50)

RAW = "../data/raw/2026-10-12_132605/"
DEPARTEMENTS = ["59", "69", "75"]

datasets = {
    d: {"dpe_existants": pd.read_parquet(RAW + f"{d}/dpe_existants.parquet"),
        "dpe_neufs": pd.read_parquet(RAW + f"{d}/dpe_neufs.parquet")}
    for d in DEPARTEMENTS
}
# Fusion seule (224 colonnes, aucun nettoyage), comme l'ancien fichier processed
dfs = DataPreprocessor(appliquer_nettoyage=False).process(datasets)

for d, df in dfs.items():
    print(d, df.shape)

59 (140344, 226)
69 (144964, 224)
75 (108644, 229)


## 2. Comparaison des trois départements
Paris compte moins de lignes que les deux autres départements. On vérifie la répartition neufs / existants et les colonnes manquantes.

In [4]:
# Répartition neufs / existants par département
for d, df in dfs.items():
    print(d, df["origine"].value_counts().to_dict())

# Colonnes qui ne sont pas présentes dans tous les départements
toutes = set().union(*[set(df.columns) for df in dfs.values()])
for d, df in dfs.items():
    absentes = sorted(toutes - set(df.columns))
    print(f"\n{d} : {len(absentes)} colonne(s) absente(s) -> {absentes}")

59 {'dpe_existants': 100000, 'dpe_neufs': 40344}
69 {'dpe_existants': 100000, 'dpe_neufs': 44964}
75 {'dpe_existants': 100000, 'dpe_neufs': 8644}

59 : 7 colonne(s) absente(s) -> ['conso_ef_generateur_n2_ecs_n1', 'date_installation_generateur_n2_ecs_n1', 'description_generateur_n2_ecs_n1', 'type_energie_generateur_n2_ecs_n1', 'type_generateur_n2_ecs_n1', 'usage_generateur_n2_ecs_n1', 'volume_stockage_generateur_n2_ecs_n1']

69 : 9 colonne(s) absente(s) -> ['conso_ef_generateur_n2_ecs_n1', 'date_installation_generateur_n2_ecs_n1', 'description_generateur_n2_ecs_n1', 'facteur_couverture_solaire_installation_chauffage_n2', 'facteur_couverture_solaire_saisi_installation_chauffage_n2', 'type_energie_generateur_n2_ecs_n1', 'type_generateur_n2_ecs_n1', 'usage_generateur_n2_ecs_n1', 'volume_stockage_generateur_n2_ecs_n1']

75 : 4 colonne(s) absente(s) -> ['facteur_couverture_solaire_installation_chauffage_n2', 'facteur_couverture_solaire_saisi_installation_chauffage_n1', 'facteur_couverture_so

**Observation (étape 2)**

- **Paris** a 8 644 logements neufs, contre 40 344 dans le Nord et 44 964 dans le Rhône. Les neufs sont sous le plafond de 100 000 : on a donc sans doute tous les DPE neufs.
- Les logements existants sont plafonnés à **100 000 par département** (plafond atteint partout), alors que les neufs sont sous le plafond : la part des neufs varie beaucoup (environ 8 % à Paris, 29 à 31 % dans le Nord et le Rhône). Ce n'est pas la réalité du parc : c'est l'effet du plafond de récupération. Il ne faut donc pas en tirer de conclusion sur le département.
- Les départements n'ont pas exactement les mêmes colonnes (voir la sortie ci-dessus). Ce n'est pas un problème : le nettoyage traite chaque département séparément.

## 3. Vue d'ensemble par département

Pour chaque département, on regarde : les doublons, la période couverte et la répartition des étiquettes DPE.

In [5]:
for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print("Doublons sur numero_dpe :", df["numero_dpe"].duplicated().sum())

    dates = pd.to_datetime(df["date_reception_dpe"], errors="coerce")
    print("Période :", dates.min().date(), "->", dates.max().date())

    print("Étiquettes :")
    print(pd.crosstab(df["origine"], df["etiquette_dpe"]))
    print()

===== Département 59 =====
Doublons sur numero_dpe : 0
Période : 2021-07-01 -> 2026-10-05
Étiquettes :
etiquette_dpe     A     B      C      D      E     F     G
origine                                                   
dpe_existants   352  1260  26312  36586  22102  8783  4605
dpe_neufs      7473  8286  24439    142      2     0     2

===== Département 69 =====
Doublons sur numero_dpe : 0
Période : 2021-07-01 -> 2026-10-05
Étiquettes :
etiquette_dpe     A      B      C      D      E     F     G
origine                                                    
dpe_existants   481   2520  36459  33401  17451  6134  3554
dpe_neufs      7914  10028  26838    145      7     1    31

===== Département 75 =====
Doublons sur numero_dpe : 0
Période : 2021-07-01 -> 2026-09-14
Étiquettes :
etiquette_dpe     A     B      C      D      E      F      G
origine                                                     
dpe_existants   138   553  12318  29290  31525  14693  11483
dpe_neufs      3021  2333   32

**Observations (étape 3)**
- Aucun doublon dans les trois départements.
- Les logements neufs sont presque tous en A, B ou C.
- Les logements existants de Paris sont plus souvent en E que ceux de Lille et de Lyon.
- Les neufs en G sont presque tous à Lyon (12), très peu à Lille (1) et aucun à Paris.
- Lille et Lyon s'arrêtent au 09/02/2026, Paris va jusqu'au 14/09/2026 : à vérifier, nos données ne couvrent peut-être pas toute la période.

## 4. Répartition des DPE dans le temps
On regarde combien de DPE il y a par année et par type, pour voir si l'échantillon couvre bien toute la période.

In [6]:
for d, df in dfs.items():
    dates = pd.to_datetime(df["date_reception_dpe"], errors="coerce")
    print(f"===== Département {d} =====")
    print(pd.crosstab(dates.dt.year, df["origine"]))
    print("Dernière date par type :")
    print(dates.groupby(df["origine"]).max().dt.date)
    print()

===== Département 59 =====
origine             dpe_existants  dpe_neufs
date_reception_dpe                          
2021                        24096       2928
2022                        69151       9733
2023                           74       9612
2024                          501       9697
2025                          309       5089
2026                         5869       3285
Dernière date par type :
origine
dpe_existants    2026-02-09
dpe_neufs        2026-10-05
Name: date_reception_dpe, dtype: object

===== Département 69 =====
origine             dpe_existants  dpe_neufs
date_reception_dpe                          
2021                        19541       4255
2022                        68620      12540
2023                         5504      10083
2024                          431       9125
2025                          595       5225
2026                         5309       3736
Dernière date par type :
origine
dpe_existants    2026-02-09
dpe_neufs        2026-10-05
Name: d

**Observations (étape 4)**
- Existants de Lille et Lyon : environ la moitié des DPE sont de 2021, l'autre moitié de 2026, et presque aucun entre les deux.
- Existants de Paris : presque tous de 2026.
- Lille et Lyon s'arrêtent au 09/02/2026 : il manque les mois suivants.
- Conséquence : l'échantillon n'est pas représentatif dans le temps. On ne peut pas étudier l'évolution dans le temps avec ces données.

## 5. Les DPE de 2021 et de 2026 sont-ils comparables ?
Les règles de calcul du DPE ont pu changer entre les deux périodes. On compare, pour les logements existants, la répartition des étiquettes et la consommation médiane par année.

In [7]:
for d, df in dfs.items():
    ex = df[df["origine"] == "dpe_existants"].copy()
    ex["annee"] = pd.to_datetime(ex["date_reception_dpe"], errors="coerce").dt.year
    print(f"===== Département {d} (existants) =====")

    # Part de chaque étiquette par année
    print(pd.crosstab(ex["annee"], ex["etiquette_dpe"], normalize="index").round(2))

    # Consommation médiane par année, pour les 3 énergies de chauffage les plus fréquentes
    top = ex["type_energie_principale_chauffage"].value_counts().head(3).index
    print(ex[ex["type_energie_principale_chauffage"].isin(top)]
          .pivot_table(index="annee", columns="type_energie_principale_chauffage",
                       values="conso_5_usages_par_m2_ep", aggfunc="median").round(0))
    print()

===== Département 59 (existants) =====
etiquette_dpe     A     B     C     D     E     F     G
annee                                                  
2021           0.00  0.01  0.23  0.35  0.24  0.10  0.06
2022           0.00  0.01  0.27  0.37  0.22  0.09  0.04
2023           0.00  0.01  0.38  0.41  0.16  0.04  0.00
2024           0.00  0.01  0.57  0.24  0.16  0.02  0.00
2025           0.01  0.13  0.42  0.23  0.09  0.12  0.01
2026           0.01  0.03  0.30  0.40  0.17  0.06  0.02
type_energie_principale_chauffage  Gaz naturel  Réseau de Chauffage urbain  \
annee                                                                        
2021                                     224.0                       170.0   
2022                                     212.0                       177.0   
2023                                     136.0                       288.0   
2024                                     142.0                       252.0   
2025                                     164.

**Observations (étape 5)**
- Pour les logements existants, les étiquettes et la consommation médiane de 2021 et 2026 sont comparables dans chaque département : on garde toutes les années.
- Les années 2022 à 2025 ont trop peu de lignes (moins de 400) pour être interprétées.
- Paris est différent de Lille et de Lyon : plus de logements en E, F et G, et consommation électrique plus élevée. C'est une vraie différence du parc, pas une erreur.
- À mentionner dans le rapport : nos données sont concentrées sur 2021 et 2026, elles ne permettent pas d'étudier l'évolution dans le temps car cela dessinerait deux pics sur 2021 et 2026.

## 6. Consommation et valeurs aberrantes
Variable : `conso_5_usages_par_m2_ep` (kWh/m²/an). On cherche les valeurs impossibles ou extrêmes pour fixer un seuil, en regardant chaque département.

In [8]:
col = "conso_5_usages_par_m2_ep"

for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print(df[col].quantile([0.01, 0.5, 0.9, 0.99, 0.999, 1]).round(1))
    print("Valeurs <= 0  :", (df[col] <= 0).sum())
    print("Valeurs > 800 :", (df[col] > 800).sum())
    print("5 plus fortes valeurs :")
    print(df.nlargest(5, col)[["etiquette_dpe", col, "surface_habitable_logement", "origine"]])
    print()

===== Département 59 =====
0.010      32.6
0.500     172.0
0.900     322.3
0.990     526.1
0.999     762.4
1.000    2625.6
Name: conso_5_usages_par_m2_ep, dtype: float64
Valeurs <= 0  : 0
Valeurs > 800 : 112
5 plus fortes valeurs :
      etiquette_dpe  conso_5_usages_par_m2_ep  surface_habitable_logement  \
26751             G                    2625.6                        56.0   
5813              G                    2014.3                        82.1   
1124              G                    1869.0                        34.4   
36765             G                    1854.9                        36.6   
1215              G                    1853.3                        53.6   

             origine  
26751  dpe_existants  
5813   dpe_existants  
1124   dpe_existants  
36765  dpe_existants  
1215   dpe_existants  

===== Département 69 =====
0.010         36.0
0.500        149.3
0.900        297.1
0.990        508.2
0.999        747.1
1.000    3855376.2
Name: conso_5_usages_par_

**Observations (étape 6)**
- Aucune consommation nulle ou négative.
- Peu de valeurs au-dessus de 800 kWh/m²/an : 17 à Lille, 14 à Lyon, 42 à Paris (environ 0,12 % des lignes).
- Paris a des consommations plus élevées (médiane de 148 contre environ 95 ailleurs), ce qui est cohérent avec son parc ancien et ses petites surfaces.
- Deux cas parmi les valeurs extrêmes : des très petites surfaces (8 à 20 m²), où une forte consommation par m² est possible, et des surfaces normales (environ 80 m²) avec plus de 1 800 kWh/m²/an, très probablement des erreurs de saisie.

## 7. Choix du seuil maximum de consommation
On compare plusieurs seuils : combien de lignes supprimerait chacun, et à quelles surfaces correspondent les valeurs extrêmes.

In [9]:
col = "conso_5_usages_par_m2_ep"
surf = "surface_habitable_logement"

# 1. Combien de lignes seraient supprimées selon le seuil ?
lignes = []
for d, df in dfs.items():
    for seuil in [600, 800, 1000, 1200]:
        n = (df[col] > seuil).sum()
        lignes.append({"departement": d, "seuil": seuil,
                       "lignes_supprimees": n,
                       "part_%": round(n / len(df) * 100, 2)})
pd.DataFrame(lignes).pivot(index="seuil", columns="departement",
                           values=["lignes_supprimees", "part_%"])

lignes_supprimees                part_%            
departement                59     69      75     59    69    75
seuil                                                          
600                     638.0  564.0  3767.0   0.45  0.39  3.47
800                     112.0   99.0   758.0   0.08  0.07  0.70
1000                     26.0   21.0   158.0   0.02  0.01  0.15
1200                     19.0   11.0    49.0   0.01  0.01  0.05

In [10]:
# 2. À quelles surfaces correspondent les valeurs > 800 ?
for d, df in dfs.items():
    extremes = df[df[col] > 800]
    print(f"===== Département {d} : {len(extremes)} valeurs > 800 =====")
    print(pd.cut(extremes[surf], [0, 9, 20, 50, 1000]).value_counts().sort_index())
    print("Surface manquante :", extremes[surf].isna().sum())
    print()

===== Département 59 : 112 valeurs > 800 =====
surface_habitable_logement
(0, 9]         2
(9, 20]       41
(20, 50]      35
(50, 1000]    33
Name: count, dtype: int64
Surface manquante : 1

===== Département 69 : 99 valeurs > 800 =====
surface_habitable_logement
(0, 9]         6
(9, 20]       38
(20, 50]      31
(50, 1000]    23
Name: count, dtype: int64
Surface manquante : 1

===== Département 75 : 758 valeurs > 800 =====
surface_habitable_logement
(0, 9]        207
(9, 20]       338
(20, 50]      164
(50, 1000]     43
Name: count, dtype: int64
Surface manquante : 6



**Observations (étape 7)**
- Un seuil à 600 kWh/m²/an est trop sévère : il supprimerait de vraies passoires thermiques (la classe G commence vers 420 kWh/m²/an).
- Au-dessus de 1 000, il ne reste que quelques valeurs isolées (8 à Lille, 3 à Lyon, 7 à Paris), très probablement des erreurs de saisie.
- À Paris, presque toutes les consommations supérieures à 800 concernent des logements de moins de 20 m², plausibles pour de très petits logements.
- À Lille et à Lyon, quelques valeurs supérieures à 800 concernent des logements de plus de 50 m² : peu crédibles.

**Décision proposée** : seuil maximum de consommation de **1 000 kWh/m²/an**. Il supprime 18 lignes sur 58 644 (0,03 %) et conserve les petits logements parisiens plausibles.

## 8. Surface et type de bâtiment
La règle sur la surface est liée à celle sur la consommation. On regarde les surfaces trop petites ou trop grandes, les surfaces manquantes, et le type de bâtiment concerné.

In [11]:
surf = "surface_habitable_logement"

for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print(df[surf].describe().round(1))
    print("Surface manquante :", df[surf].isna().sum())
    print("Surface < 9 m²    :", (df[surf] < 9).sum())
    print("Surface > 300 m²  :", (df[surf] > 300).sum())
    print("Surface > 500 m²  :", (df[surf] > 500).sum())
    print("Hauteur sous plafond :")
    print(df["hauteur_sous_plafond"].describe().round(2))
    print("Type de bâtiment, quand la surface est manquante :")
    print(df.loc[df[surf].isna(), "type_batiment"].value_counts())
    print()

===== Département 59 =====
count    139366.0
mean         79.6
std         124.6
min           6.5
25%          48.0
50%          68.3
75%          91.2
max       17486.0
Name: surface_habitable_logement, dtype: float64
Surface manquante : 978
Surface < 9 m²    : 16
Surface > 300 m²  : 750
Surface > 500 m²  : 417
Hauteur sous plafond :
count    140344.00
mean          2.61
std           5.70
min           1.10
25%           2.50
50%           2.50
75%           2.50
max        1998.00
Name: hauteur_sous_plafond, dtype: float64
Type de bâtiment, quand la surface est manquante :
type_batiment
immeuble       776
maison         135
appartement     67
Name: count, dtype: int64

===== Département 69 =====
count    143447.0
mean         73.6
std         116.8
min           7.3
25%          44.6
50%          63.7
75%          82.7
max       11298.8
Name: surface_habitable_logement, dtype: float64
Surface manquante : 1517
Surface < 9 m²    : 21
Surface > 300 m²  : 836
Surface > 500 m²  : 537
Ha

**Observations (étape 8)**
- Des surfaces impossibles : maximum de 10 078 m² (Lille), 6 317 m² (Lyon) et 101 036 m² (Paris). Environ 100 à 125 lignes par département dépassent 500 m².
- Les surfaces manquantes sont presque toutes des DPE de type `immeuble` (410 sur 412 à Paris). Un DPE d'immeuble décrit un bâtiment entier, pas un logement.
- La hauteur sous plafond est parfois saisie en centimètres au lieu de mètres (maximum de 310 à Lille et 270 à Lyon, alors que la médiane est de 2,5).
- Paris compte beaucoup de logements de moins de 9 m² (environ 1 170 sur le snapshot actuel) : presque tous sont de vrais petits logements (chambres de service, studios), pas des erreurs. Seule une cinquantaine est sous 5 m², ce qui est suspect.

## 9. Vérifications avant de fixer les règles
On regarde si les très grandes surfaces sont des immeubles, comment se répartissent les petites surfaces, et combien de hauteurs sous plafond sont en centimètres.

In [12]:
surf = "surface_habitable_logement"

for d, df in dfs.items():
    print(f"===== Département {d} =====")

    # Surface par type de bâtiment
    print(df.groupby("type_batiment")[surf].agg(["count", "median", "max"]).round(1))

    # Type de bâtiment des surfaces > 300 m²
    print("Surfaces > 300 m², par type :", df.loc[df[surf] > 300, "type_batiment"].value_counts().to_dict())

    # Répartition des petites surfaces
    print("Petites surfaces :", pd.cut(df[surf], [0, 5, 7, 9]).value_counts().sort_index().to_dict())
    print()

===== Département 59 =====
               count  median      max
type_batiment                        
appartement    80390    53.1   2192.3
immeuble         469  1167.0  17486.0
maison         58507    92.8   1965.0
Surfaces > 300 m², par type : {'immeuble': 399, 'maison': 317, 'appartement': 34}
Petites surfaces : {Interval(0, 5, closed='right'): 0, Interval(5, 7, closed='right'): 2, Interval(7, 9, closed='right'): 19}

===== Département 69 =====
                count  median      max
type_batiment                         
appartement    122915    60.2   7457.5
immeuble          676   903.1  11298.8
maison          19856   110.1   1651.0
Surfaces > 300 m², par type : {'immeuble': 579, 'maison': 217, 'appartement': 40}
Petites surfaces : {Interval(0, 5, closed='right'): 0, Interval(5, 7, closed='right'): 0, Interval(7, 9, closed='right'): 28}

===== Département 75 =====
                count  median       max
type_batiment                          
appartement    106123    41.5    393

**Observations (étape 9)**
- Les immeubles ont une surface médiane d'environ 1 000 m² (contre 40 à 110 m² pour les appartements et les maisons). Ils expliquent la plupart des surfaces supérieures à 300 m² et presque toutes les surfaces manquantes.
- Il reste quelques erreurs de saisie parmi les logements (par exemple un appartement de 3 932 m² à Paris).
- La hauteur sous plafond est parfois saisie en centimètres (270, 310) ou multipliée par 10 (25, 26). Il y a 119 valeurs supérieures à 5 m sur 58 644 lignes.
- Pour savoir si ces hauteurs concernent surtout les immeubles, on a comparé avant et après exclusion des immeubles (résultat ci-dessous).

**Décisions provisoires**
- Exclure les DPE de type `immeuble` (1 271 lignes, soit 2,2 %, sur le snapshot précédent ; environ 4 130 lignes, soit 1 %, sur le snapshot actuel) : l'application concerne des logements, pas des bâtiments entiers.
- Supprimer les lignes sans surface.
- Surface minimale : 5 m² (voir l'étape 10). Surface maximale : à fixer (étape 10).
- Consommation maximale : 1 000 kWh/m²/an.

In [13]:
h = "hauteur_sous_plafond"

for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print("Hauteur < 1.8 m       :", (df[h] < 1.8).sum())
    print("Hauteur entre 1.8 et 5 :", df[h].between(1.8, 5).sum())
    print("Hauteur > 5           :", (df[h] > 5).sum())
    print("Valeurs > 5 (extrait) :", sorted(df.loc[df[h] > 5, h].unique())[:15])
    print()

===== Département 59 =====
Hauteur < 1.8 m       : 8
Hauteur entre 1.8 et 5 : 139740
Hauteur > 5           : 596
Valeurs > 5 (extrait) : [np.float64(5.1), np.float64(5.2), np.float64(5.3), np.float64(5.4), np.float64(5.5), np.float64(5.6), np.float64(5.7), np.float64(5.8), np.float64(5.9), np.float64(6.0), np.float64(6.1), np.float64(6.2), np.float64(6.3), np.float64(6.4), np.float64(6.5)]

===== Département 69 =====
Hauteur < 1.8 m       : 78
Hauteur entre 1.8 et 5 : 144402
Hauteur > 5           : 484
Valeurs > 5 (extrait) : [np.float64(5.1), np.float64(5.2), np.float64(5.3), np.float64(5.4), np.float64(5.5), np.float64(5.6), np.float64(5.7), np.float64(5.8), np.float64(5.9), np.float64(6.0), np.float64(6.1), np.float64(6.4), np.float64(6.6), np.float64(6.8), np.float64(6.9)]

===== Département 75 =====
Hauteur < 1.8 m       : 25
Hauteur entre 1.8 et 5 : 108553
Hauteur > 5           : 66
Valeurs > 5 (extrait) : [np.float64(5.1), np.float64(5.2), np.float64(5.3), np.float64(5.4), np.fl

In [14]:
h = "hauteur_sous_plafond"

for d, df in dfs.items():
    hautes = df[df[h] > 5]
    print(f"===== Département {d} : {len(hautes)} valeurs > 5 m =====")
    print(hautes["type_batiment"].value_counts().to_dict())

    sans_immeuble = df[df["type_batiment"] != "immeuble"]
    print("Hors immeubles :", (sans_immeuble[h] > 5).sum(), "valeurs > 5 m")
    print()

===== Département 59 : 596 valeurs > 5 m =====
{'maison': 347, 'appartement': 226, 'immeuble': 23}
Hors immeubles : 573 valeurs > 5 m

===== Département 69 : 484 valeurs > 5 m =====
{'appartement': 433, 'maison': 32, 'immeuble': 19}
Hors immeubles : 465 valeurs > 5 m

===== Département 75 : 66 valeurs > 5 m =====
{'appartement': 55, 'maison': 9, 'immeuble': 2}
Hors immeubles : 64 valeurs > 5 m



**Résultat (hauteur sous plafond)**
- Sur 119 valeurs supérieures à 5 m, seulement 4 concernent des immeubles : 115 concernent des appartements et des maisons (85 à Lille, 27 à Lyon, 3 à Paris).
- Exclure les immeubles ne suffit donc pas pour corriger cette variable.

**Décision (hauteur sous plafond)** : les valeurs hors de 1,8 à 5 m sont remplacées par « manquant ». On ne supprime pas la ligne et on ne devine pas l'unité de saisie (mètres ou centimètres). Le modèle remplacera le manquant par une valeur typique (la médiane) au moment de l'entraînement.

## 10. Surface maximale et coordonnées
On regarde la répartition des grandes surfaces parmi les logements (hors immeubles), puis le format des coordonnées pour la carte.

In [15]:
surf = "surface_habitable_logement"

for d, df in dfs.items():
    logements = df[df["type_batiment"] != "immeuble"]
    print(f"===== Département {d} : {len(logements)} logements (hors immeubles) =====")
    print(pd.cut(logements[surf], [0, 9, 200, 300, 400, 500, 1000, 1e6]).value_counts().sort_index())
    print()

===== Département 59 : 139099 logements (hors immeubles) =====
surface_habitable_logement
(0.0, 9.0]                 21
(9.0, 200.0]           136926
(200.0, 300.0]           1599
(300.0, 400.0]            248
(400.0, 500.0]             47
(500.0, 1000.0]            45
(1000.0, 1000000.0]        11
Name: count, dtype: int64

===== Département 69 : 143069 logements (hors immeubles) =====
surface_habitable_logement
(0.0, 9.0]                 28
(9.0, 200.0]           141179
(200.0, 300.0]           1307
(300.0, 400.0]            172
(400.0, 500.0]             48
(500.0, 1000.0]            25
(1000.0, 1000000.0]        12
Name: count, dtype: int64

===== Département 75 : 107654 logements (hors immeubles) =====
surface_habitable_logement
(0.0, 9.0]               1271
(9.0, 200.0]           105389
(200.0, 300.0]            604
(300.0, 400.0]            144
(400.0, 500.0]             24
(500.0, 1000.0]            19
(1000.0, 1000000.0]        11
Name: count, dtype: int64



In [16]:
for d, df in dfs.items():
    print(d, "-> coordonnées manquantes :", df["_geopoint"].isna().sum())

print()
print(dfs["69"]["_geopoint"].head())
print(type(dfs["69"]["_geopoint"].iloc[0]))

59 -> coordonnées manquantes : 0
69 -> coordonnées manquantes : 0
75 -> coordonnées manquantes : 0

0    46.001749036869164,4.710939968070691
1     45.77587496570502,4.918295955089466
2     45.74137898096509,4.861241989402495
3     45.74496000928658,4.863709027217594
4     45.77250803193388,4.880935964606717
Name: _geopoint, dtype: str
<class 'str'>


**Observations (étape 10)**
- Hors immeubles, plus de 97 % des logements ont une surface entre 9 et 200 m².
- De 200 à 500 m², ce sont de grandes maisons plausibles (environ 280 lignes au total).
- Au-dessus de 500 m², il ne reste que 17 lignes sur environ 57 000 (8 à Lille, 3 à Lyon, 6 à Paris) : erreurs de saisie probables.
- Paris compte environ 1 170 logements de moins de 9 m², dont seulement une cinquantaine sous 5 m².
- La colonne `_geopoint` est un texte au format « latitude,longitude », sans valeur manquante.

**Décisions (surface)**
- Surface minimale : **5 m²**. Surface maximale : 500 m². Un seuil à 9 m² retirait à Paris environ 1 170 logements, presque tous réels : on le descend à 5 m².
- Les coordonnées sont extraites en découpant `_geopoint` au niveau de la virgule.
## 11. Coordonnées et neufs classés D, E ou G
On vérifie que les coordonnées sont convertibles et situées en France, puis on regarde les logements neufs avec une mauvaise étiquette.

In [17]:
for d, df in dfs.items():
    coords = df["_geopoint"].str.split(",", expand=True)
    lat = pd.to_numeric(coords[0], errors="coerce")
    lon = pd.to_numeric(coords[1], errors="coerce")

    print(f"===== Département {d} =====")
    print("Non convertibles :", lat.isna().sum())
    print("Latitude  :", lat.min().round(2), "->", lat.max().round(2))
    print("Longitude :", lon.min().round(2), "->", lon.max().round(2))
    print("Hors de la France métropolitaine :",
          (~(lat.between(41, 52) & lon.between(-5.5, 10))).sum())
    print()

===== Département 59 =====
Non convertibles : 0
Latitude  : -5.98 -> 51.08
Longitude : -1.36 -> 4.2
Hors de la France métropolitaine : 214

===== Département 69 =====
Non convertibles : 0
Latitude  : -5.98 -> 46.29
Longitude : -1.36 -> 5.13
Hors de la France métropolitaine : 554

===== Département 75 =====
Non convertibles : 0
Latitude  : -5.98 -> 48.9
Longitude : -1.36 -> 2.42
Hors de la France métropolitaine : 204



In [18]:
cols = ["etiquette_dpe", "etiquette_ges", "conso_5_usages_par_m2_ep",
        "emission_ges_5_usages_par_m2", "surface_habitable_logement",
        "type_energie_principale_chauffage"]

for d, df in dfs.items():
    neufs_ge = df[(df["origine"] == "dpe_neufs") & df["etiquette_dpe"].isin(["E", "G"])]
    print(f"===== Département {d} : {len(neufs_ge)} neufs en E ou G =====")
    if len(neufs_ge):
        print(neufs_ge[cols].sort_values("etiquette_dpe"))
    print()

===== Département 59 : 4 neufs en E ou G =====
       etiquette_dpe etiquette_ges  conso_5_usages_par_m2_ep  \
113575             E             E                     164.8   
129717             E             B                     270.3   
107650             G             G                     639.0   
132745             G             F                    1851.6   

        emission_ges_5_usages_par_m2  surface_habitable_logement  \
113575                          61.1                        80.5   
129717                           9.8                       255.6   
107650                         119.6                        73.7   
132745                         103.4                        19.1   

       type_energie_principale_chauffage  
113575                       Électricité  
129717                       Électricité  
107650                       Gaz naturel  
132745                       Électricité  

===== Département 69 : 38 neufs en E ou G =====
       etiquette_dpe etique

**Observations (étape 11)**
- Coordonnées : 22 points invalides à Lille et 184 à Lyon, tous situés à la même position (latitude -5,98, longitude -1,36), en plein océan. Probablement une valeur par défaut après un échec de géocodage. Paris est cohérent.
- Neufs en G à Lyon (12 lignes) : consommation de 58 à 83 kWh/m²/an (niveau A ou B) mais émissions de GES de 5 900 à 8 600 kg CO₂/m²/an, ce qui est impossible. L'étiquette G vient donc d'une erreur de saisie des émissions. Tous sont chauffés par un réseau de chauffage urbain.
- Neuf en G à Lille (1 ligne) et neufs en E (3 lignes) : étiquettes cohérentes avec la consommation, on les garde.

**Décisions (étape 11)**
- Coordonnées hors de la France métropolitaine : latitude et longitude remplacées par « manquant » (la ligne est conservée).
- Émissions de GES aberrantes : lignes supprimées (seuil à fixer).

## 12. Vérifications : coordonnées invalides et émissions de GES
On vérifie que les coordonnées invalides sont bien un échec de géocodage, puis on fixe le seuil des émissions de GES aberrantes.

In [19]:
for d, df in dfs.items():
    coords = df["_geopoint"].str.split(",", expand=True)
    lat = pd.to_numeric(coords[0], errors="coerce")
    lon = pd.to_numeric(coords[1], errors="coerce")
    invalide = ~(lat.between(41, 52) & lon.between(-5.5, 10))

    print(f"===== Département {d} : {invalide.sum()} coordonnées invalides =====")
    print(df.loc[invalide, "_geopoint"].value_counts().head(3))
    print("Statut de géocodage :", df.loc[invalide, "statut_geocodage"].value_counts(dropna=False).to_dict())
    print("Type de logement    :", df.loc[invalide, "origine"].value_counts().to_dict())
    print()

===== Département 59 : 214 coordonnées invalides =====
_geopoint
-5.98385630920877,-1.363081210117898    214
Name: count, dtype: int64
Statut de géocodage : {"adresse géocodée ban à l'adresse": 214}
Type de logement    : {'dpe_existants': 214}

===== Département 69 : 554 coordonnées invalides =====
_geopoint
-5.98385630920877,-1.363081210117898     553
-5.983569780356801,-1.363065199959244      1
Name: count, dtype: int64
Statut de géocodage : {"adresse géocodée ban à l'adresse": 554}
Type de logement    : {'dpe_existants': 554}

===== Département 75 : 204 coordonnées invalides =====
_geopoint
-5.98385630920877,-1.363081210117898      193
-5.983551234198215,-1.3630832623914177      3
-5.9835511508586245,-1.36308318418074       2
Name: count, dtype: int64
Statut de géocodage : {"adresse géocodée ban à l'adresse": 204}
Type de logement    : {'dpe_existants': 204}



In [20]:
e = "emission_ges_5_usages_par_m2"

for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print(df[e].quantile([0.5, 0.99, 0.999, 1]).round(1).to_dict())
    print("Émissions > 200 :", (df[e] > 200).sum())
    print(df[df[e] > 200].groupby(["origine", "etiquette_dpe"]).size())
    print()

===== Département 59 =====
{0.5: 16.0, 0.99: 106.0, 0.999: 172.4, 1.0: 843.0}
Émissions > 200 : 55
origine        etiquette_dpe
dpe_existants  G                55
dtype: int64

===== Département 69 =====
{0.5: 14.0, 0.99: 94.2, 0.999: 153.0, 1.0: 871997.3}
Émissions > 200 : 54
origine        etiquette_dpe
dpe_existants  G                23
dpe_neufs      G                31
dtype: int64

===== Département 75 =====
{0.5: 21.0, 0.99: 128.0, 0.999: 218.0, 1.0: 1645.0}
Émissions > 200 : 165
origine        etiquette_dpe
dpe_existants  G                165
dtype: int64



**Observations (étape 12)**
- Coordonnées invalides : 22 lignes à Lille et 184 à Lyon, toutes des logements existants, avec exactement la même position (latitude -5,98 ; longitude -1,36). Le statut de géocodage indique pourtant « adresse géocodée », donc l'erreur vient des données de l'ADEME et pas d'un échec visible. Paris n'est pas concerné.
- Émissions de GES : médiane d'environ 13 kg CO₂/m²/an et 99,9 % des logements sous 175. Les maximums plausibles sont de 456 (Lille) et 320 (Paris).
- À Lyon, les 12 neufs classés G ont des émissions de 5 900 à 8 600 : valeurs impossibles qui produisent une étiquette fausse.

**Décisions (étape 12)**
- Coordonnées hors de la France métropolitaine : latitude et longitude remplacées par « manquant ». La ligne est conservée et ne s'affiche pas sur la carte.
- Émissions de GES supérieures à 1 000 kg CO₂/m²/an : lignes supprimées (environ 12 lignes à Lyon). Cette vérification devra être refaite après application des règles.

## 13. Choix des colonnes : méthode (exploration)

> **Évolution** : cette sélection de 30 colonnes a servi à l'exploration. Le nettoyage final **ne sélectionne plus de colonnes** : toutes sont conservées, sauf celles dont plus de 80 % des valeurs sont manquantes (section 15 et `docs/nettoyage_donnees.md`). `docs/colonnes_retenues.csv` reste la liste des colonnes déjà examinées, utilisée par l'audit des colonnes. Les colonnes qui contiennent la réponse restent à exclure des variables d'entrée des modèles.

Sur les 224 colonnes, on ne garde que celles utiles à l'application.

**Critères de conservation** : une colonne est gardée si elle sert à identifier un DPE, à filtrer, à calculer un KPI ou un graphique, à placer un point sur la carte, ou à alimenter un modèle.

**Critères de suppression** : colonne technique (identifiants internes de l'API), trop détaillée ou trop vide (équipements secondaires, copropriété), résultat intermédiaire du calcul du DPE (déperditions, besoins, apports), ou adresse précise (inutile : la carte utilise le point géocodé).

**Cas particulier** : les colonnes qui contiennent déjà la réponse (consommations par usage, émissions de GES, coût) sont gardées pour l'analyse mais **exclues des modèles**.

**Limites** : le sens de chaque colonne est déduit de son nom. À confirmer dans le dictionnaire de données de l'ADEME. Les colonnes que les règles ne couvrent pas sont listées « à examiner » et décidées en équipe.

Le tableau complet (224 colonnes, décision et raison) est enregistré dans `docs/colonnes_retenues.csv`.

Une seule cellule applique toutes les décisions sur les 224 colonnes. Pour chaque colonne, la décision est prise dans cet ordre :
1. **Décision manuelle** (colonne par colonne, après examen des valeurs manquantes et des valeurs présentes).
2. **Règles générales** (par rôle ou par motif dans le nom de la colonne).
3. Sinon, la colonne est marquée « à examiner ».

Résultat : `docs/colonnes_retenues.csv` (colonne, groupe, décision, raison, taux de valeurs manquantes) et deux listes, `COLONNES_GARDEES` et `COLONNES_ANALYSE`.

In [21]:
import os
import re

toutes = sorted(set().union(*[set(df.columns) for df in dfs.values()]))

# ---------------------------------------------------------------
# 1. Colonnes gardées, par rôle
# ---------------------------------------------------------------
IDENT = {"numero_dpe", "date_reception_dpe", "origine"}
CIBLES = {"etiquette_dpe", "conso_5_usages_par_m2_ep"}
GEO = {"code_departement_ban", "code_postal_ban", "nom_commune_ban", "code_insee_ban", "_geopoint"}
LOGEMENT = {"surface_habitable_logement", "periode_construction", "type_batiment",
            "nombre_niveau_logement", "hauteur_sous_plafond"}
CHAUFFAGE = {"type_energie_principale_chauffage", "type_installation_chauffage",
             "type_energie_principale_ecs", "type_installation_ecs"}
ISOLATION = {"qualite_isolation_enveloppe", "qualite_isolation_murs",
             "qualite_isolation_menuiseries", "qualite_isolation_plancher_bas"}
# Analyse seulement : contiennent déjà la réponse ou sont des résultats de calcul
ANALYSE = {"etiquette_ges", "emission_ges_5_usages_par_m2", "conso_5_usages_ep",
           "conso_chauffage_ep", "conso_ecs_ep", "conso_eclairage_ep",
           "conso_auxiliaires_ep", "cout_total_5_usages", "ubat_w_par_m2_k"}

# ---------------------------------------------------------------
# 2. Décisions manuelles : colonnes supprimées après examen
# ---------------------------------------------------------------
SUPPRESSIONS = [
    (["annee_construction"], "Redondante",
     "remplacée par periode_construction, complète"),
    (["zone_climatique", "classe_altitude"], "Redondante",
     "manquantes à 45-71 % chez les neufs ; le département porte l'information"),
    (["categorie_enr", "electricite_pv_autoconsommee", "nombre_module", "presence_production_pv",
      "surface_totale_capteurs_pv", "systeme_production_electricite_origine_renouvelable"],
     "Trop incomplète", "panneaux solaires et énergies renouvelables : manquantes à 65 % ou plus"),
    (["surface_climatisee", "type_energie_climatisation"],
     "Trop incomplète", "climatisation : manquantes à 98 % ou plus"),
    (["type_ventilation", "surface_ventilee"],
     "Trop incomplète", "ventilation : manquantes à 95 % ou plus"),
    (["indicateur_confort_ete", "isolation_toiture", "logement_traversant",
      "presence_brasseur_air", "protection_solaire_exterieure"],
     "Trop incomplète", "manquantes à 46 % chez les existants et à 90 % ou plus chez les neufs"),
    (["classe_inertie_batiment", "inertie_lourde"],
     "Trop incomplète", "manquantes à 96 % ou plus chez les neufs"),
    (["nombre_appartement", "nombre_niveau_immeuble", "numero_etage_appartement",
      "position_logement_dans_immeuble", "typologie_logement", "appartement_non_visite"],
     "Immeuble / diagnostic", "information sur l'immeuble ou le diagnostic, manquante à plus de 50 % pour un type"),
    (["qualite_isolation_plancher_haut_comble_amenage", "qualite_isolation_plancher_haut_comble_perdu",
      "qualite_isolation_plancher_haut_toit_terrasse"],
     "Trop incomplète", "isolation du plancher haut : manquantes à 54 % ou plus"),
    (["date_derniere_modification_dpe", "date_etablissement_dpe",
      "date_fin_validite_dpe", "date_visite_diagnostiqueur"],
     "Redondante", "date_reception_dpe suffit pour le suivi et le rafraîchissement"),
    (["coordonnee_cartographique_x_ban", "coordonnee_cartographique_y_ban"],
     "Redondante", "la carte utilise _geopoint (latitude et longitude)"),
    (["version_dpe"], "Redondante", "reflète la date de réception ; inconnue du particulier"),
    (["modele_dpe"], "Quasi constante",
     "une seule valeur chez les existants, plus de 96 % de RT2012 chez les neufs"),
    (["methode_application_dpe"], "Redondante", "recoupe origine et type_batiment"),
    (["ventilation_posterieure_2012"], "Quasi constante", "96 % à 99 % de 0"),
    (["production_electricite_pv_kwhep_par_an"], "Trop incomplète",
     "94 % à 96 % de zéros, valeurs aberrantes"),
    (["nombre_niveau_logement"], "Incohérente",
     "plus de 3 niveaux pour 29 % des appartements neufs, contre 2-3 % chez les existants ; manquante à 14-45 % chez les neufs"),
    (["type_energie_principale_ecs"], "Redondante",
     "« Non affecté » pour 100 % des neufs : recoupe origine"),
]
DECISIONS_MANUELLES = {c: (groupe, "supprimée", raison)
                       for cols, groupe, raison in SUPPRESSIONS for c in cols}

# ---------------------------------------------------------------
# 3. Règles générales (la première qui correspond décide)
# ---------------------------------------------------------------
REGLES = [
    ("Technique", "supprimée", lambda c: c in {"_id", "_i", "_rand", "_score"},
     "identifiant interne de l'API, sans information métier"),
    ("Identification", "gardée", lambda c: c in IDENT, "suivi des DPE, filtre neuf/existant, rafraîchissement"),
    ("Cible", "gardée", lambda c: c in CIBLES, "variable à analyser et à prédire"),
    ("Géographie", "gardée", lambda c: c in GEO, "carte et filtres géographiques"),
    ("Logement", "gardée", lambda c: c in LOGEMENT, "filtres, graphiques et modèles"),
    ("Chauffage / eau chaude", "gardée", lambda c: c in CHAUFFAGE, "filtres, graphiques et modèles"),
    ("Isolation", "gardée", lambda c: c in ISOLATION, "analyse, modèles à décider"),
    ("Analyse seulement", "gardée (hors modèles)", lambda c: c in ANALYSE,
     "contient déjà la réponse ou résultat de calcul : KPI et graphiques, pas les modèles"),
    ("Adresse détaillée", "supprimée",
     lambda c: bool(re.search(r"adresse|nom_rue|numero_voie|complement_adresse|identifiant_ban|score_ban|statut_geocodage|id_rnb|code_region_ban|_brut$", c)),
     "adresse précise, inutile : la carte utilise le point géocodé"),
    ("Résultat de calcul", "supprimée",
     lambda c: bool(re.match(r"(conso_|emission_ges_|cout_|deperditions_|besoin_|apport_)", c)),
     "résultat intermédiaire du calcul du DPE, détail non utile à l'app"),
    ("Équipement détaillé", "supprimée",
     lambda c: bool(re.search(r"_n[123]\b|_n[123]_|_installation_|generateur|description_", c)),
     "détail technique des équipements, très incomplet"),
    ("Copropriété / immeuble", "supprimée",
     lambda c: bool(re.search(r"copropriete|rpls|dpe_immeuble|dpe_remplace|nom_residence|surface_.*immeuble|surface_tertiaire", c)),
     "information sur le bâtiment ou la copropriété, hors sujet"),
]

def classer(col):
    """Renvoie (groupe, décision, raison) : décision manuelle d'abord, puis règles générales."""
    if col in DECISIONS_MANUELLES:
        return DECISIONS_MANUELLES[col]
    for groupe, decision, test, raison in REGLES:
        if test(col):
            return groupe, decision, raison
    return "À examiner", "à examiner", "non couverte par les règles : à décider en équipe"

# ---------------------------------------------------------------
# 4. Tableau final : décision, raison, taux de valeurs manquantes
# ---------------------------------------------------------------
manq = {}
for origine in ["dpe_existants", "dpe_neufs"]:
    par_dept = []
    for d, df in dfs.items():
        g = df[(df["type_batiment"] != "immeuble") & (df["origine"] == origine)]
        par_dept.append(g.reindex(columns=toutes).isna().mean() * 100)
    manq[origine] = pd.concat(par_dept, axis=1).max(axis=1).round(0)

tableau = pd.DataFrame([(c, *classer(c)) for c in toutes],
                       columns=["colonne", "groupe", "decision", "raison"])
tableau["manq_existants_%"] = tableau["colonne"].map(manq["dpe_existants"])
tableau["manq_neufs_%"] = tableau["colonne"].map(manq["dpe_neufs"])

print(tableau["decision"].value_counts(), "\n")
print("Encore à examiner :", tableau.loc[tableau["decision"] == "à examiner", "colonne"].tolist())

COLONNES_GARDEES = tableau.loc[tableau["decision"].str.startswith("gardée"), "colonne"].tolist()
COLONNES_ANALYSE = tableau.loc[tableau["decision"] == "gardée (hors modèles)", "colonne"].tolist()
print(len(COLONNES_GARDEES), "colonnes gardées, dont", len(COLONNES_ANALYSE), "hors modèles")

os.makedirs("../docs", exist_ok=True)
#tableau.to_csv("../docs/colonnes_retenues.csv", index=False, encoding="utf-8-sig")

decision
supprimée                203
gardée                    21
gardée (hors modèles)      9
Name: count, dtype: int64 

Encore à examiner : []
30 colonnes gardées, dont 9 hors modèles


**Bilan du choix des colonnes** (historique de l'exploration, plus appliqué par le nettoyage final)
Sur 224 colonnes, **30 sont conservées** et 194 supprimées. Le détail (colonne, groupe, décision, raison, taux de valeurs manquantes) est dans `docs/colonnes_retenues.csv`.

| Groupe | Nombre | Usage |
|---|---|---|
| Identification | 3 | suivi, filtre neuf/existant |
| Cibles | 2 | variables à analyser et à prédire |
| Géographie | 5 | carte et filtres |
| Logement | 5 | filtres, graphiques, modèles |
| Chauffage et eau chaude | 4 | filtres, graphiques, modèles |
| Isolation | 4 | analyse, modèles à décider |
| Analyse seulement | 9 | KPI et graphiques, **exclues des modèles** |

Les colonnes d'analyse contiennent déjà la réponse (consommations, émissions de GES, coût) ou sont des résultats de calcul du DPE (`ubat_w_par_m2_k`). Elles seront listées dans `COLONNES_ANALYSE` pour être écartées des variables d'entrée des modèles.

## 14. Harmonisation des catégories
Pour chaque colonne de catégories conservée, on vérifie les valeurs présentes dans les trois départements : orthographe, casse, espaces, catégories rares ou incohérentes.

In [22]:
categorielles = [
    "periode_construction", "type_batiment",
    "type_energie_principale_chauffage", "type_energie_principale_ecs",
    "type_installation_chauffage", "type_installation_ecs",
    "qualite_isolation_enveloppe", "qualite_isolation_murs",
    "qualite_isolation_menuiseries", "qualite_isolation_plancher_bas",
    "nombre_niveau_logement",
]

for col in categorielles:
    valeurs = pd.concat([df[col] for df in dfs.values()]).dropna().astype(str)
    print(f"===== {col} : {valeurs.nunique()} valeurs distinctes =====")

    tab = pd.DataFrame({d: df[col].value_counts(dropna=False) for d, df in dfs.items()})
    print(tab.fillna(0).astype(int).sort_values("59", ascending=False).head(12))

    # Même valeur écrite de plusieurs façons (casse ou espaces) ?
    if valeurs.str.strip().str.lower().nunique() < valeurs.nunique():
        print("Une même valeur est écrite de plusieurs façons (casse ou espaces)")
    print()

===== periode_construction : 10 valeurs distinctes =====
                         59     69     75
periode_construction                     
avant 1948            40300  30641  67888
après 2021            30087  34004   6982
1948-1974             28703  30255  20423
2013-2021             13397  17006   3065
1978-1982              5999   3342   1937
2006-2012              5968   9910    724
1989-2000              5940   9206   3644
1975-1977              3950   3582   2096
1983-1988              3673   3368   1095
2001-2005              2327   3650    790

===== type_batiment : 3 valeurs distinctes =====
                  59      69      75
type_batiment                       
appartement    80457  123057  106312
maison         58642   20012    1342
immeuble        1245    1895     990

===== type_energie_principale_chauffage : 15 valeurs distinctes =====
                                                       59     69     75
type_energie_principale_chauffage                            

**Observations (étape 14)**
- Aucune valeur écrite de plusieurs façons (casse, espaces) : pas de correction nécessaire. Un `strip()` sera tout de même appliqué dans le prétraitement pour les données futures de l'API.
- `periode_construction` : 10 périodes cohérentes entre les trois départements. Un ordre chronologique sera défini pour les filtres.
- `type_batiment` : 3 valeurs (appartement, maison, immeuble). Les immeubles (1 271 lignes) sont exclus.
- `type_energie_principale_chauffage` : 13 valeurs, dont plusieurs très rares (charbon : 6 lignes, GPL : 39, électricité d'origine renouvelable : 16).
- `type_installation_chauffage` et `type_installation_ecs` sont manquantes sur les mêmes lignes.
- `qualite_isolation_*` : 4 niveaux ordonnés (insuffisante, moyenne, bonne, très bonne).
- `nombre_niveau_logement` : valeurs jusqu'à 16 niveaux, peu plausibles pour un seul logement. À vérifier.
- `type_energie_principale_ecs` : la valeur « Non affecté » est la plus fréquente (environ 70 % des lignes à Lille et à Lyon, 48 % à Paris). Son sens est à vérifier.

**Décisions envisagées (étape 14) — non retenues dans le nettoyage final** : le nettoyage conserve les valeurs d'origine et les cases vides (aucun regroupement d'énergies, aucun « inconnu » ni « non applicable ») ; ces choix seront à refaire à la modélisation.
- Énergie de chauffage : les 4 variantes « Bois – … » sont regroupées en « Bois » ; GPL, charbon, propane et électricité d'origine renouvelable en « Autre ». Les valeurs inconnues futures vont dans « Autre ».
- `type_installation_chauffage` et `type_installation_ecs` : manquants remplacés par « inconnu ».
- `qualite_isolation_*` : conservées avec leur ordre ; manquants traités dans le pipeline des modèles.
- `nombre_niveau_logement` et `type_energie_principale_ecs` : à décider après vérification.

In [23]:
# 1. Toutes les valeurs de l'énergie de chauffage (le tableau précédent s'arrêtait à 12)
print(pd.concat([df["type_energie_principale_chauffage"] for df in dfs.values()]).value_counts(), "\n")

for d, df in dfs.items():
    logements = df[df["type_batiment"] != "immeuble"]
    niv = logements["nombre_niveau_logement"]
    print(f"===== Département {d} (hors immeubles) =====")

    # 2. Part des logements avec plus de 3 niveaux, ou sans valeur, par type
    print(logements.assign(plus_de_3_niveaux=niv > 3, niveaux_manquants=niv.isna())
                   .groupby(["origine", "type_batiment"])[["plus_de_3_niveaux", "niveaux_manquants"]]
                   .mean().round(2))

    # 3. Installation de chauffage manquante, par type
    print(logements.assign(installation_manquante=logements["type_installation_chauffage"].isna())
                   .groupby(["origine", "type_batiment"])["installation_manquante"].mean().round(2))

    # 4. Eau chaude « Non affecté », par type de logement
    print(logements.assign(ecs_non_affecte=logements["type_energie_principale_ecs"].str.startswith("Non aff", na=False))
                   .groupby("origine")["ecs_non_affecte"].mean().round(2))
    print()

type_energie_principale_chauffage
Électricité                                                     182370
Gaz naturel                                                     162752
Réseau de Chauffage urbain                                       33408
Fioul domestique                                                  5766
Non affecté                                                       3217
Bois – Granulés (pellets) ou briquettes                           2517
Bois – Bûches                                                     2278
GPL                                                                667
Bois – Plaquettes forestières                                      484
Charbon                                                            256
Bois – Plaquettes d’industrie                                       95
Électricité d'origine renouvelable utilisée dans le bâtiment        70
Propane                                                             68
Butane                                     

In [24]:
# Ordres des catégories ordonnées
ORDRE_PERIODE = ["avant 1948", "1948-1974", "1975-1977", "1978-1982", "1983-1988",
                 "1989-2000", "2001-2005", "2006-2012", "2013-2021", "après 2021"]
ORDRE_ISOLATION = ["insuffisante", "moyenne", "bonne", "très bonne"]

def regrouper_energie(valeur):
    """Regroupe les énergies de chauffage rares ou proches."""
    if pd.isna(valeur):
        return "Inconnu"
    v = str(valeur).strip()
    if v.startswith("Bois"):
        return "Bois"
    if v in {"Électricité", "Gaz naturel", "Réseau de Chauffage urbain",
             "Fioul domestique", "Non affecté"}:
        return v
    return "Autre"

# Contrôle sur les trois départements (sans modifier les données)
for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print(df["type_energie_principale_chauffage"].map(regrouper_energie).value_counts().to_dict())
    print("Périodes hors liste :", set(df["periode_construction"].dropna()) - set(ORDRE_PERIODE))
    print("Installation chauffage :", df["type_installation_chauffage"].fillna("non applicable").value_counts().to_dict())
    print()

===== Département 59 =====
{'Gaz naturel': 70375, 'Électricité': 57078, 'Réseau de Chauffage urbain': 5620, 'Bois': 3022, 'Fioul domestique': 2061, 'Non affecté': 1662, 'Autre': 526}
Périodes hors liste : set()
Installation chauffage : {'individuel': 60897, 'non applicable': 59265, 'collectif': 18829, 'mixte (collectif-individuel)': 1353}

===== Département 69 =====
{'Électricité': 64158, 'Gaz naturel': 60108, 'Réseau de Chauffage urbain': 14816, 'Fioul domestique': 2175, 'Bois': 2145, 'Non affecté': 1263, 'Autre': 299}
Périodes hors liste : set()
Installation chauffage : {'individuel': 88856, 'collectif': 33022, 'non applicable': 20986, 'mixte (collectif-individuel)': 2100}

===== Département 75 =====
{'Électricité': 61134, 'Gaz naturel': 32269, 'Réseau de Chauffage urbain': 12972, 'Fioul domestique': 1530, 'Non affecté': 292, 'Autre': 240, 'Bois': 207}
Périodes hors liste : set()
Installation chauffage : {'individuel': 69890, 'collectif': 33781, 'mixte (collectif-individuel)': 3416, 

## 15. Application du prétraitement
Les règles décidées dans ce notebook sont écrites dans `src/preprocessing.py`. On les applique à chaque département séparément (mêmes règles) et on affiche le bilan : nombre de lignes après chaque étape. Le nettoyage ne sélectionne pas de colonnes : il retire seulement celles vides à plus de 80 %, il ne remplace aucune valeur manquante et il exclut les immeubles.

In [25]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2
from src.preprocessing import DataPreprocessor

RAW = "../data/raw/2026-10-12_132605/"
datasets = {
    d: {"dpe_existants": pd.read_parquet(RAW + f"{d}/dpe_existants.parquet"),
        "dpe_neufs": pd.read_parquet(RAW + f"{d}/dpe_neufs.parquet")}
    for d in DEPARTEMENTS
}

prepro = DataPreprocessor()
propres = prepro.process(datasets)

print({d: df.shape for d, df in propres.items()})
pd.concat([b.rename(columns={"lignes": d}) for d, b in prepro.bilans.items()], axis=1)

{'59': (138807, 159), '69': (142675, 159), '75': (107235, 156)}


,59,69,75
etape,,,
Données brutes,140344,144964,108644
Conversion des types,140344,144964,108644
Doublons supprimés,140344,144964,108644
Immeubles exclus,139099,143069,107654
Manquants clés supprimés,138889,142763,107462
Étiquettes A-G,138889,142763,107462
Surface 5-500 m²,138833,142726,107385
Consommation ≤ 1000,138807,142705,107235
Émissions GES ≤ 1000,138807,142675,107235


In [26]:
for d, df in propres.items():
    print(f"===== {d} =====")
    print(pd.crosstab(df["origine"], df["etiquette_dpe"]))
    print(df[["surface_habitable_logement", "conso_5_usages_par_m2_ep",
              "emission_ges_5_usages_par_m2"]].describe().round(1).loc[["min", "50%", "max"]])
    print("Coordonnées manquantes :", df["latitude"].isna().sum(),
          "| hauteur manquante :", df["hauteur_sous_plafond"].isna().sum())
    print()

===== 59 =====
etiquette_dpe     A     B      C      D      E     F     G
origine                                                   
dpe_existants   351  1245  26136  36293  21914  8686  4536
dpe_neufs      7398  8175  23930    142      1     0     0
     surface_habitable_logement  conso_5_usages_par_m2_ep  \
min                         6.5                       5.9   
50%                        68.1                     172.5   
max                       500.0                     997.9   

     emission_ges_5_usages_par_m2  
min                           0.0  
50%                          16.0  
max                         302.0  
Coordonnées manquantes : 212 | hauteur manquante : 579

===== 69 =====
etiquette_dpe     A     B      C      D      E     F     G
origine                                                   
dpe_existants   414  2497  36070  33090  17272  6072  3496
dpe_neufs      7667  9774  26178    140      5     0     0
     surface_habitable_logement  conso_5_usages_par_m

In [27]:
for d, etiq in [("59", "G"), ("75", "E")]:
    n = datasets[d]["dpe_neufs"]
    n = n[n["etiquette_dpe"] == etiq]
    print(f"===== {d}, neufs en {etiq} =====")
    print(n[["numero_dpe", "type_batiment", "surface_habitable_logement",
             "conso_5_usages_par_m2_ep", "emission_ges_5_usages_par_m2"]])

===== 59, neufs en G =====
          numero_dpe type_batiment  surface_habitable_logement  \
7650   2259N1081433G      immeuble                        73.7   
32745  2359N2590135H   appartement                        19.1   

       conso_5_usages_par_m2_ep  emission_ges_5_usages_par_m2  
7650                      639.0                         119.6  
32745                    1851.6                         103.4  
===== 75, neufs en E =====
         numero_dpe type_batiment  surface_habitable_logement  \
5570  2275N0713911K      immeuble                       181.7   

      conso_5_usages_par_m2_ep  emission_ges_5_usages_par_m2  
5570                     289.7                          36.8  


In [28]:
tous = pd.concat(propres.values(), keys=propres.keys(), names=["dept"])

# 1. Valeurs manquantes qui restent, en %
manq = (tous.isna().mean() * 100).round(1)
print(manq[manq > 0].sort_values(ascending=False), "\n")

# 2. Étiquette GES : valeurs présentes
print(tous["etiquette_ges"].value_counts(dropna=False).sort_index(), "\n")

# 3. Valeurs extrêmes des colonnes d'analyse
cols = ["conso_5_usages_ep", "conso_chauffage_ep", "conso_ecs_ep", "conso_eclairage_ep",
        "conso_auxiliaires_ep", "cout_total_5_usages", "ubat_w_par_m2_k"]
print(tous[cols].describe(percentiles=[0.5, 0.99, 0.999]).round(1).T[["min", "50%", "99%", "99.9%", "max"]], "\n")

# 4. Cohérence (hypothèse : conso_5_usages_ep est la consommation totale annuelle)
ecart = (tous["conso_5_usages_ep"] / tous["surface_habitable_logement"]
         - tous["conso_5_usages_par_m2_ep"]).abs()
print("Lignes où l'écart dépasse 5 kWh/m²/an :", int((ecart > 5).sum()))

qualite_isolation_plancher_haut_comble_amenage         91.9
categorie_enr                                          91.2
nombre_niveau_immeuble                                 76.2
appartement_non_visite                                 73.9
complement_adresse_batiment                            70.2
surface_habitable_immeuble                             57.6
qualite_isolation_plancher_haut_comble_perdu           56.1
inertie_lourde                                         52.3
isolation_toiture                                      49.5
logement_traversant                                    49.0
protection_solaire_exterieure                          49.0
presence_brasseur_air                                  49.0
indicateur_confort_ete                                 47.2
id_rnb                                                 46.4
provenance_id_rnb                                      46.4
surface_ventilee                                       42.8
type_ventilation                        

In [29]:
# 1. Les 128 lignes où conso_5_usages_ep / surface ne retombe pas sur la cible
tous["ecart"] = ecart
bizarres = tous[ecart > 5]
print(bizarres.groupby(["dept", "origine"]).size(), "\n")
print(bizarres[["surface_habitable_logement", "conso_5_usages_par_m2_ep",
                "conso_5_usages_ep", "ecart"]].describe().round(1).loc[["min", "50%", "max"]], "\n")

# 2. Part des auxiliaires dans la consommation totale
part_aux = tous["conso_auxiliaires_ep"] / tous["conso_5_usages_ep"]
print(part_aux.describe(percentiles=[0.5, 0.99, 0.999]).round(2), "\n")
tous.loc[part_aux > 0.5, ["origine", "surface_habitable_logement", "etiquette_dpe",
                          "conso_5_usages_par_m2_ep", "conso_auxiliaires_ep"]].head(8)

dept  origine      
59    dpe_existants    729
      dpe_neufs          5
69    dpe_existants    620
      dpe_neufs          3
75    dpe_existants    918
dtype: int64 

     surface_habitable_logement  conso_5_usages_par_m2_ep  conso_5_usages_ep  \
min                         5.0                      66.7              929.5   
50%                        53.0                     280.0             8623.1   
max                       371.1                     999.0           285427.4   

      ecart  
min     5.0  
50%    96.1  
max  3864.5   

count    388717.00
mean          0.07
std           0.17
min           0.00
50%           0.03
99%           1.00
99.9%         2.30
max           2.30
dtype: float64 



origine  surface_habitable_logement etiquette_dpe  \
dept                                                                 
59   5081  dpe_existants                        72.0             C   
     5110  dpe_existants                       110.0             C   
     5111  dpe_existants                        20.2             G   
     5125  dpe_existants                       120.0             G   
     5133  dpe_existants                       120.0             B   
     5135  dpe_existants                        89.8             E   
     5148  dpe_existants                        20.1             G   
     5151  dpe_existants                        73.2             G   

           conso_5_usages_par_m2_ep  conso_auxiliaires_ep  
dept                                                       
59   5081                      53.0                3822.9  
     5110                     140.7               15480.5  
     5111                     500.4               10119.8  
     5125                     100.9               12119.8  
     5133                      86.1               10335.0  
     5135                     199.9               17964.7  
     5148                     137.3                2758.5  
     5151                     146.0               10695.6

In [30]:
usages = ["conso_chauffage_ep", "conso_ecs_ep", "conso_eclairage_ep", "conso_auxiliaires_ep"]
somme = tous[usages].sum(axis=1)

# Le total des usages ne peut pas dépasser la consommation totale (le refroidissement n'est pas gardé)
incoh = somme > tous["conso_5_usages_ep"] * 1.01
part_aux = tous["conso_auxiliaires_ep"] / tous["conso_5_usages_ep"]

print("Somme des usages > total :", int(incoh.sum()), "lignes")
print("Auxiliaires > total      :", int((part_aux > 1).sum()), "lignes\n")
print(tous[incoh].groupby(["dept", "origine"]).size(), "\n")
print(tous.loc[part_aux > 1, ["origine", "conso_5_usages_ep", "conso_auxiliaires_ep"]].head(8))

Somme des usages > total : 291054 lignes
Auxiliaires > total      : 2571 lignes

dept  origine      
59    dpe_existants    78534
      dpe_neufs        26586
69    dpe_existants    81812
      dpe_neufs        27141
75    dpe_existants    71318
      dpe_neufs         5663
dtype: int64 

                 origine  conso_5_usages_ep  conso_auxiliaires_ep
dept                                                             
59   5208  dpe_existants            11175.0               11644.9
     5239  dpe_existants             9043.4               20799.7
     5254  dpe_existants            12350.5               16914.2
     5481  dpe_existants            13209.1               15050.2
     5495  dpe_existants            22351.2               41764.3
     5498  dpe_existants            26943.1               30638.2
     5505  dpe_existants             6506.0                8529.2
     5620  dpe_existants             4006.3                9214.6


In [31]:
bilan = pd.concat({d: b["lignes"] for d, b in prepro.bilans.items()}, axis=1)
bilan["total"] = bilan.sum(axis=1)
bilan.to_csv("../docs/bilan_nettoyage.csv", encoding="utf-8-sig")
bilan

,59,69,75,total
etape,,,,
Données brutes,140344,144964,108644,393952
Conversion des types,140344,144964,108644,393952
Doublons supprimés,140344,144964,108644,393952
Immeubles exclus,139099,143069,107654,389822
Manquants clés supprimés,138889,142763,107462,389114
Étiquettes A-G,138889,142763,107462,389114
Surface 5-500 m²,138833,142726,107385,388944
Consommation ≤ 1000,138807,142705,107235,388747
Émissions GES ≤ 1000,138807,142675,107235,388717


## 16. Conclusion du nettoyage

### 16.1 Données de départ
- 3 départements (59, 69, 75), snapshot `2026-10-12_132605` : **393 952 lignes** brutes et de 182 à 224 colonnes selon le département et le type (existants ou neufs).
- 100 000 logements existants par département (plafond atteint partout). Neufs : 8 644 à Paris, 40 344 dans le Nord, 44 964 dans le Rhône.
- Logements neufs et existants fusionnés par département, distingués par la colonne `origine`.
- Aucun doublon sur `numero_dpe`.
- Après nettoyage : **388 717 logements** (98,7 %), avec 156 colonnes pour Paris et 159 pour le Nord et le Rhône.

### 16.2 Limites des données (à reprendre dans le rapport ML)
- **Échantillon plafonné** : 100 000 logements existants par département. Les neufs sont sous le plafond, donc probablement complets.
- **La part des neufs varie beaucoup** (environ 8 % à Paris, 29 à 31 % dans le Nord et le Rhône) : c'est un effet du plafond, pas la composition du parc.
- **Pas forcément un tirage aléatoire** : sur le snapshot précédent (10 000 lignes par type), les existants du Nord et du Rhône étaient concentrés sur 2021 et 2026, ceux de Paris sur 2026. À revérifier sur le snapshot actuel avant toute conclusion sur l'évolution dans le temps.
- **Étiquettes déséquilibrées** : le C domine, F et G sont rares, les neufs sont presque tous en A, B ou C.
- **Les départements diffèrent** : Paris a davantage de E, F et G et des consommations plus élevées (consommation médiane d'environ 263 kWh/m²/an contre 173 dans le Nord et 150 dans le Rhône, après nettoyage), à cause d'un parc ancien et de petites surfaces. C'est une vraie différence, pas une erreur.
- **Le sens de chaque colonne est déduit de son nom** : à confirmer dans le dictionnaire de données de l'ADEME.

### 16.3 Règles qui suppriment des lignes
| Règle | Seuil | Justification | Lignes retirées (Nord / Rhône / Paris) |
|---|---|---|---|
| Doublons | `numero_dpe` | éviter de compter deux fois un DPE | 0 / 0 / 0 |
| Immeubles exclus | `type_batiment` = immeuble | un DPE d'immeuble décrit un bâtiment entier (surface médiane d'environ 1 000 m²) ; l'application concerne un logement. Interrupteur : `TYPES_BATIMENT_EXCLUS` dans `src/preprocessing.py` | 1 245 / 1 895 / 990 |
| Manquants clés | numéro, étiquette, consommation ou surface absents | variables indispensables | 210 / 306 / 192 |
| Étiquette | A à G | étiquettes valides | 0 / 0 / 0 |
| Surface | de 5 à 500 m² | en dessous de 5 m², ce n'est pas un logement ; entre 5 et 9 m², ce sont de vrais petits logements (Paris : environ 1 170 sous 9 m²), à garder ; au-delà de 500 m², erreurs de saisie probables | 56 / 37 / 77 |
| Consommation | supérieure à 0 et au plus 1 000 kWh/m²/an | la classe G commence vers 420 ; un seuil à 600 supprimerait de vraies passoires thermiques ; au-dessus de 1 000, quelques valeurs isolées | 26 / 21 / 150 |
| Émissions de GES | au plus 1 000 kg CO₂/m²/an | les valeurs plausibles vont jusqu'à environ 460 ; les neufs de Lyon classés G ont des émissions de 5 900 à 8 600 : étiquette fausse | 0 / 30 / 0 |

Au total, **5 235 lignes retirées sur 393 952 (1,3 %)**.

- **Colonnes de consommation par usage non additives** : sur le snapshot précédent, la somme des 4 usages (chauffage, eau chaude, éclairage, auxiliaires) dépassait la consommation totale pour 51 % des lignes (environ 69 % des neufs et 34 % des existants). La proportion est trop élevée pour des erreurs de saisie : c'est probablement une question de définition des colonnes, à vérifier dans le dictionnaire de données de l'ADEME. Les cellules ci-dessus refont le calcul sur le snapshot actuel.
- **Décision** : aucune ligne supprimée pour cette raison. La cible `conso_5_usages_par_m2_ep` est valide. Les colonnes par usage ne sont ni additionnées ni exprimées en parts : les KPI et graphiques utilisent la consommation par m².

### 16.4 Règles qui conservent la ligne
| Règle | Justification |
|---|---|
| Coordonnées hors de la France métropolitaine → manquantes | sur le snapshot précédent, 206 logements existants (22 à Lille, 184 à Lyon) partageaient la même position erronée, en plein océan ; le nombre actuel est dans la sortie « Coordonnées manquantes » de la section 15 |
| Hauteur sous plafond hors de 1,8 à 5 m → manquante | unité de saisie erronée probable (270 ou 310 en cm, 25 ou 26 multipliées par 10) ; le nombre actuel est dans la sortie de la section 15 |
| Espaces supprimés dans les textes | protection pour les futures données de l'API |

- **Valeurs manquantes conservées** : le nettoyage ne remplace aucune case vide (ni médiane, ni « inconnu », ni « non applicable ») et ne supprime pas les lignes incomplètes, sauf si une information indispensable manque. Le remplissage se fera à la modélisation, sur l'entraînement seulement, pour éviter qu'une information du jeu de test serve à préparer l'entraînement.
- **Exclusion des immeubles** : un DPE classé « immeuble » peut parfois décrire un logement isolé (par exemple un neuf de Lille classé G et un neuf de Paris classé E sont dans ce cas). Ils sont exclus avec les autres immeubles (4 130 lignes, soit 1,0 %) ; la perte est faible et la règle protège l'application des bâtiments entiers. À confirmer par l'équipe.

### 16.5 Colonnes : toutes conservées, sauf les presque vides
- Le nettoyage **ne sélectionne pas de variables**. Il retire les colonnes dont plus de 80 % des valeurs sont manquantes (66 à 74 colonnes selon le département : deuxième installation de chauffage, troisième énergie, panneaux solaires, climatisation, copropriété, etc.). Les colonnes indispensables ne sont jamais retirées.
- Les nouvelles colonnes d'un rafraîchissement entrent automatiquement dans les données ; `src/audit_colonnes.py` permet de les repérer et signale celles qui contiennent probablement la réponse.
- **Colonnes à exclure des variables d'entrée des modèles** (elles contiennent la réponse ou sont des résultats du calcul du DPE) : les 9 colonnes d'analyse ci-dessous, et plus largement toutes celles dont le nom commence par `conso_`, `cout_`, `emission_ges`, `etiquette_`, `deperditions`, `besoin`, `ubat`.

Colonnes identifiées comme utiles à l'application lors de l'exploration :

| Groupe | Colonnes | Usage |
|---|---|---|
| Identification (3) | `numero_dpe`, `date_reception_dpe`, `origine` | suivi, filtre neuf / existant, rafraîchissement |
| Cibles (2) | `etiquette_dpe`, `conso_5_usages_par_m2_ep` | variables à analyser et à prédire |
| Géographie (5) | département, code postal, commune, code INSEE, `_geopoint` | carte et filtres |
| Logement (3) | surface, période de construction, type de bâtiment | filtres, graphiques, modèles |
| Hauteur sous plafond (1) | `hauteur_sous_plafond` | à décider pour les modèles |
| Chauffage et eau chaude (3) | énergie de chauffage, type d'installation de chauffage et d'eau chaude | filtres, graphiques, modèles |
| Isolation (4) | enveloppe, murs, menuiseries, plancher bas | analyse, modèles à décider |
| Analyse seulement (9) | étiquette GES, émissions de GES, consommations par usage, coût, `ubat_w_par_m2_k` | KPI et graphiques, **exclues des modèles** |

Le détail colonne par colonne de l'exploration est dans `docs/colonnes_retenues.csv`.

### 16.6 Ce qui n'est pas décidé ici : étape de modélisation
- **Choix des variables d'entrée**, à partir de l'exploration et en excluant les colonnes qui contiennent la réponse. À décider : `hauteur_sous_plafond`, les qualités d'isolation (un particulier ne les connaît pas forcément), `origine`, le département, la localisation.
- **Valeurs manquantes** : à remplir dans le pipeline scikit-learn, sur le jeu d'entraînement seulement. Elles concernent maintenant toutes les colonnes qui ont des cases vides, pas seulement la hauteur, les coordonnées et quelques isolations.
- **Encodage** : catégories ordonnées (période de construction, qualités d'isolation) et non ordonnées ; regroupement des modalités rares (par exemple les énergies de chauffage).
- **Découpe entraînement / test** : par département ou non, organisation des dossiers, à décider en équipe.
- **Déséquilibre des classes** : découpage stratifié, pondération des classes, métriques F1 macro et matrice de confusion.
- **Évaluation séparée** : à la fois sur l'ensemble et sur les logements existants seuls, car les neufs gonflent les performances.
- **Sensibilité des seuils** : refaire les modèles avec un seuil de consommation de 800 pour vérifier que les résultats ne changent pas.

### 16.7 Points à confirmer
- Le sens des colonnes dans le dictionnaire de données de l'ADEME.
- Auprès de l'équipe : l'exclusion des immeubles, le seuil de 80 % de valeurs manquantes, et le fait de régénérer `data/processed` à la demande plutôt que de le stocker dans Git.
- L'extraction pourrait-elle utiliser un tirage aléatoire (colonne `_rand`) plutôt que les premiers résultats ?
